# RUNECLAW — Merge & Deploy on Colab

Merges two fine-tuned models (Colab/Sonnet + Local/Haiku), converts to GGUF, and runs Ollama — all on Colab.

**Setup:** Runtime → Change runtime type → **L4 GPU** (recommended) or T4

**Requirements:** Both model safetensors uploaded to Google Drive

## 1. Setup & Mount Drive

In [ ]:
%%capture
!pip install mergekit pyyaml safetensors
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
print("Dependencies installed.")

In [ ]:
import os, json, glob, shutil, subprocess, time
import torch

from google.colab import drive
drive.mount('/content/drive')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

## 2. Locate Models on Drive

Update these paths to match your Google Drive folder names.

In [ ]:
# ════════════════════════════════════════════════
#  UPDATE THESE PATHS to match your Drive folders
# ════════════════════════════════════════════════
COLAB_MODEL = "/content/drive/MyDrive/runeclaw-model-v2"       # Sonnet-trained (4 shards, from Colab)
LOCAL_MODEL = "/content/drive/MyDrive/runeclaw-model-merged"   # Haiku-trained (8 shards, from local ASUS)
# ════════════════════════════════════════════════

def scan_model(path, label):
    print(f"\n{'='*50}")
    print(f"  {label}: {path}")
    print(f"{'='*50}")
    if not os.path.exists(path):
        print(f"  NOT FOUND! Check the path.")
        return False
    files = os.listdir(path)
    safetensors = sorted([f for f in files if f.endswith('.safetensors')])
    total_gb = sum(os.path.getsize(f"{path}/{f}") for f in safetensors) / 1024**3
    has_config = 'config.json' in files
    is_adapter = 'adapter_config.json' in files
    print(f"  Safetensors: {len(safetensors)} files ({total_gb:.1f} GB)")
    print(f"  config.json: {'YES' if has_config else 'MISSING'}")
    print(f"  Type: {'LoRA adapter' if is_adapter else 'Full model'}")
    for f in safetensors:
        sz = os.path.getsize(f"{path}/{f}") / 1024**3
        print(f"    {f}: {sz:.2f} GB")
    if is_adapter:
        print(f"\n  WARNING: LoRA adapter detected.")
        print(f"  Will auto-merge with base model before SLERP.")
    return True

ok1 = scan_model(COLAB_MODEL, "Colab (Sonnet)")
ok2 = scan_model(LOCAL_MODEL, "Local (Haiku)")

if ok1 and ok2:
    print(f"\nBoth models found! Ready to merge.")
else:
    print(f"\nFix the paths above before continuing.")

## 3. Merge Models (SLERP)

60% Colab (Sonnet) / 40% Local (Haiku) — Sonnet data is generally higher quality.
Adjust `COLAB_WEIGHT` if you want a different ratio.

In [ ]:
import yaml

COLAB_WEIGHT = 0.6  # 60% Sonnet, 40% Haiku
MERGE_METHOD = "slerp"  # slerp, linear, or ties
OUTPUT_MERGED = "/content/runeclaw-merged"

# ── Check if models are LoRA adapters that need merging first ──
def ensure_full_model(model_path, label):
    """If model is a LoRA adapter, merge it with base model."""
    if not os.path.exists(f"{model_path}/adapter_config.json"):
        print(f"  {label}: Full model (no conversion needed)")
        return model_path

    print(f"  {label}: LoRA adapter — merging with base model...")
    from unsloth import FastLanguageModel

    # Read adapter config to find base model
    with open(f"{model_path}/adapter_config.json") as f:
        adapter_cfg = json.load(f)
    base_model = adapter_cfg.get("base_model_name_or_path", "unsloth/Meta-Llama-3.1-8B-Instruct")
    print(f"    Base: {base_model}")

    # Load and merge
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=False,  # Need full precision for merge
    )

    # Save as full model
    full_path = f"/content/{label}-full"
    model.save_pretrained(full_path)
    tokenizer.save_pretrained(full_path)
    print(f"    Saved full model: {full_path}")

    # Free memory
    del model, tokenizer
    torch.cuda.empty_cache()

    return full_path

colab_path = ensure_full_model(COLAB_MODEL, "colab")
local_path = ensure_full_model(LOCAL_MODEL, "local")

print(f"\nColab model: {colab_path}")
print(f"Local model: {local_path}")

In [ ]:
# ── Create merge config ──
config = {
    "slices": [{
        "sources": [
            {"model": colab_path, "layer_range": [0, 32]},
            {"model": local_path, "layer_range": [0, 32]},
        ]
    }],
    "merge_method": MERGE_METHOD,
    "base_model": colab_path,
    "parameters": {"t": COLAB_WEIGHT},
    "dtype": "bfloat16",
}

os.makedirs(OUTPUT_MERGED, exist_ok=True)
config_path = f"{OUTPUT_MERGED}/merge_config.yml"
with open(config_path, "w") as f:
    yaml.dump(config, f)

print(f"Merge config:")
print(f"  Method: {MERGE_METHOD.upper()}")
print(f"  Colab (Sonnet): {COLAB_WEIGHT*100:.0f}%")
print(f"  Local (Haiku):  {(1-COLAB_WEIGHT)*100:.0f}%")
print(f"\nStarting merge...\n")

!mergekit-yaml {config_path} {OUTPUT_MERGED} --allow-crimes --copy-tokenizer --lazy-unpickle

print(f"\nMerged model files:")
for fn in sorted(os.listdir(OUTPUT_MERGED)):
    fp = f"{OUTPUT_MERGED}/{fn}"
    if os.path.isfile(fp):
        sz = os.path.getsize(fp)
        print(f"  {fn}: {sz/1024**3:.2f} GB" if sz > 1024**2 else f"  {fn}")

## 4. Convert to GGUF

In [ ]:
from unsloth import FastLanguageModel

print("Loading merged model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_MERGED,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

GGUF_DIR = "/content/runeclaw-merged-gguf"
GGUF_DIR_Q8 = "/content/runeclaw-merged-gguf-q8"

# ── Export Q4_K_M (fast, ~5GB) ──
print("\nExporting Q4_K_M...")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

print(f"\nQ4_K_M output:")
for fn in sorted(os.listdir(GGUF_DIR)):
    src = f"{GGUF_DIR}/{fn}"
    if os.path.isdir(src):
        print(f"  [DIR] {fn}/")
    else:
        sz = os.path.getsize(src)
        print(f"  {fn}: {sz/1024**3:.2f} GB" if sz > 1024**2 else f"  {fn}")

# ── Export Q8_0 (higher quality, ~8GB, fits RTX 4080) ──
print("\nExporting Q8_0 (higher quality for RTX 4080)...")
try:
    model.save_pretrained_gguf(GGUF_DIR_Q8, tokenizer, quantization_method="q8_0")
    has_q8 = True
except Exception as e:
    print(f"  Q8 failed: {e}")
    has_q8 = False

# Find all GGUF files
gguf_files = glob.glob(f"{GGUF_DIR}/**/*.gguf", recursive=True)
if not gguf_files:
    gguf_files = [f"{GGUF_DIR}/{fn}" for fn in os.listdir(GGUF_DIR)
                  if os.path.isfile(f"{GGUF_DIR}/{fn}") and os.path.getsize(f"{GGUF_DIR}/{fn}") > 1024**3]

gguf_files_q8 = []
if has_q8:
    gguf_files_q8 = glob.glob(f"{GGUF_DIR_Q8}/**/*.gguf", recursive=True)
    if not gguf_files_q8:
        gguf_files_q8 = [f"{GGUF_DIR_Q8}/{fn}" for fn in os.listdir(GGUF_DIR_Q8)
                         if os.path.isfile(f"{GGUF_DIR_Q8}/{fn}") and os.path.getsize(f"{GGUF_DIR_Q8}/{fn}") > 1024**3]

print(f"\nGGUF files:")
for f in gguf_files + gguf_files_q8:
    print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1024**3:.2f} GB")

GGUF_PATH = gguf_files[0] if gguf_files else None
GGUF_NAME = os.path.basename(GGUF_PATH) if GGUF_PATH else None
print(f"\nQ4_K_M ready: {GGUF_NAME}")
if gguf_files_q8:
    print(f"Q8_0 ready:   {os.path.basename(gguf_files_q8[0])}")

## 5. Install & Start Ollama on Colab

In [ ]:
# ── Install Ollama ──
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed.")

In [ ]:
# ── Start Ollama server in background ──
import subprocess, time

# Kill any existing instance
!pkill -f ollama 2>/dev/null; sleep 1

# Start server
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
)
print(f"Ollama server PID: {proc.pid}")

# Wait for server to be ready
for i in range(30):
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print(f"Ollama ready!")
        break
    except:
        time.sleep(1)
else:
    print("Ollama failed to start. Check /tmp/ollama.log")
    !cat /tmp/ollama.log | tail -20

## 6. Create RUNECLAW Model in Ollama

In [ ]:
# ── Create Modelfile ──
SYSTEM_PROMPT = (
    "You are RUNECLAW, an AI trading analyst. You analyze cryptocurrency "
    "markets using the GetClaw Confluence Engine (12 weighted indicators: "
    "RSI-14 w=1.5, MACD w=1.0, Bollinger w=1.2, EMA Cross w=1.0, Volume "
    "Profile w=1.3, OBV w=0.8, Stoch RSI w=0.7, ADX w=1.1, Ichimoku w=0.9, "
    "VWAP w=1.4, ATR w=0.8, Fibonacci w=1.0), enforce strict risk management "
    "through 22 automated checks (all must pass, fail-closed), and generate "
    "structured trade ideas. You never execute without human confirmation. "
    "Capital preservation above all."
)

modelfile_content = f"""FROM {GGUF_PATH}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""

modelfile_path = f"{GGUF_DIR}/Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print("Creating Ollama model...")
!ollama create runeclaw -f {modelfile_path}

print("\nInstalled models:")
!ollama list

## 7. Test RUNECLAW

Run live inference against the merged model.

In [ ]:
# ── Interactive test ──
import subprocess

test_prompts = [
    "Analyze BTC/USDT on 4h for trade setups.",
    "Analyze ETH/USDT at $3800 on 1h. Return JSON.",
    "Should RUNECLAW take this SOL/USDT setup? Confluence is 0.52.",
    "Calculate position size for SOL/USDT with $25,000 portfolio in VOLATILE regime.",
    "Interpret backtest: 80 trades, 42% win rate, +0.12R expectancy, 8.5% max drawdown.",
]

for i, prompt in enumerate(test_prompts):
    print(f"\n{'='*60}")
    print(f"TEST {i+1}: {prompt}")
    print(f"{'='*60}")
    result = subprocess.run(
        ["ollama", "run", "runeclaw", prompt],
        capture_output=True, text=True, timeout=120
    )
    print(result.stdout[:1000])
    if result.stderr:
        print(f"[stderr: {result.stderr[:200]}]")

In [ ]:
# ── Custom prompt (edit and re-run) ──
CUSTOM_PROMPT = "Scan AVAX/USDT on 1h for long setups. Show full risk check."

result = subprocess.run(
    ["ollama", "run", "runeclaw", CUSTOM_PROMPT],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)

## 8. Ollama API (for bot integration testing)

Test the OpenAI-compatible API that the bot will use.

In [ ]:
# ── Test OpenAI-compatible API ──
import urllib.request, json

payload = json.dumps({
    "model": "runeclaw",
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Analyze BTC/USDT on 4h. Return JSON trade decision."}
    ],
    "temperature": 0.3,
    "stream": False,
}).encode()

req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"},
)

print("Calling Ollama API (same endpoint the bot uses)...\n")
resp = urllib.request.urlopen(req, timeout=120)
data = json.loads(resp.read())

reply = data["choices"][0]["message"]["content"]
print(reply[:1500])

# Check if JSON parse works (bot requirement)
try:
    parsed = json.loads(reply)
    print(f"\nJSON parse: SUCCESS")
    print(f"  Direction: {parsed.get('direction')}")
    print(f"  Confidence: {parsed.get('confidence')}")
    print(f"  Verdict: {parsed.get('verdict')}")
except:
    print(f"\nJSON parse: FAILED (response is text format, not JSON)")

## 9. Save Merged Model to Drive

In [ ]:
# ── Save GGUF + Modelfile to Drive ──
drive_dir = "/content/drive/MyDrive/runeclaw-final"
os.makedirs(drive_dir, exist_ok=True)

# Copy Q4_K_M files (skip directories like .cache)
for fn in os.listdir(GGUF_DIR):
    src = f"{GGUF_DIR}/{fn}"
    if os.path.isdir(src):
        print(f"  Skipping directory: {fn}")
        continue
    dst = f"{drive_dir}/{fn}"
    print(f"  Copying {fn}...")
    shutil.copy2(src, dst)

# Copy Q8_0 if available
if has_q8 and gguf_files_q8:
    drive_dir_q8 = f"{drive_dir}/q8"
    os.makedirs(drive_dir_q8, exist_ok=True)
    for fn in os.listdir(GGUF_DIR_Q8):
        src = f"{GGUF_DIR_Q8}/{fn}"
        if os.path.isdir(src):
            continue
        dst = f"{drive_dir_q8}/{fn}"
        print(f"  Copying Q8/{fn}...")
        shutil.copy2(src, dst)

    # Q8 Modelfile
    q8_name = os.path.basename(gguf_files_q8[0])
    q8_mf = f"""FROM ./{q8_name}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""
    with open(f"{drive_dir_q8}/Modelfile", "w") as f:
        f.write(q8_mf)

# Q4 Modelfile (relative path for local use)
local_modelfile = f"""FROM ./{GGUF_NAME}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""
with open(f"{drive_dir}/Modelfile", "w") as f:
    f.write(local_modelfile)

print(f"\nSaved to: {drive_dir}")
print(f"\nFiles:")
for fn in sorted(os.listdir(drive_dir)):
    fp = f"{drive_dir}/{fn}"
    if os.path.isfile(fp):
        sz = os.path.getsize(fp)
        print(f"  {fn}: {sz/1024**3:.2f} GB" if sz > 1024**2 else f"  {fn}")
    elif os.path.isdir(fp):
        print(f"  {fn}/")
        for sub in sorted(os.listdir(fp)):
            sfp = f"{fp}/{sub}"
            if os.path.isfile(sfp):
                ssz = os.path.getsize(sfp)
                print(f"    {sub}: {ssz/1024**3:.2f} GB" if ssz > 1024**2 else f"    {sub}")

print(f"\n{'='*50}")
print(f"  DEPLOY ON BTO DESKTOP")
print(f"{'='*50}")
print(f"  1. Download runeclaw-final from Google Drive")
print(f"  2. For Q4_K_M (fast, 5GB):")
print(f"     cd runeclaw-final")
print(f"     ollama create runeclaw -f Modelfile")
print(f"  3. For Q8_0 (best quality, 8GB — RTX 4080):")
print(f"     cd runeclaw-final/q8")
print(f"     ollama create runeclaw-q8 -f Modelfile")
print(f"  4. Test:")
print(f"     ollama run runeclaw 'Scan BTC/USDT on 4h'")
print(f"\n  For bot integration:")
print(f"  Set LLM_PROVIDER=ollama")
print(f"  Set LLM_MODEL=runeclaw")
print(f"  Set LLM_BASE_URL=http://localhost:11434/v1")